# NFreezer Test

In [ ]:
import nfreezer
nfreezer.backup(src='/home/ben/Desktop', dest='admin@192.168.0.25:./usbshare1-2/benoit/backup-nas/Divers/', sftppwd='toto', encryptionpwd='strongpassword')

Starting backup...
Source path: /home/ben/Desktop
Destination host: 192.168.0.25
Destination path: ./usbshare1-2/benoit/backup-nas/Divers/
Distant files list: getting...
Distant files list: done.
New, sending file: kdenlive-20.08.0-x86_64.appimage


In [3]:

def parseaddress(addr):
    if '@' in addr:
        user, r = addr.split('@', 1)  # split on first occurence
        if ':' in r and '/' not in user: # remote address. windows ok: impossible to have ':' after '@' in a path. linux: if a local dir is really named a@b.com:/hello/, use ./a@b.com:/hello/. what if '/' is in the username? technically possible with useradd, but not allowed by adduser, so evil corner case ignored here.
            host, path = r.split(':', 1)
            return True, user.strip(), host.strip(), path.strip()
    return False, None, None, addr       # not remote in all other cases


In [11]:
import pysftp
import Crypto.Protocol.KDF


def KDF(pwd, salt=None):
    if salt is None:
        salt = Crypto.Random.new().read(16)
    key = Crypto.Protocol.KDF.PBKDF2(pwd, salt, count=100*1000)
    return key, salt

src='/home/ben/Desktop'
dest='admin@192.168.0.25:./usbshare1-2/benoit/backup-nas/Divers/'
sftppwd='toto'
encryptionpwd='strongpassword'
remote, user, host, remotepath =  parseaddress(dest)
print(remote, user, host, remotepath)
if not remote or not user or not host or not remotepath:  # either not remote (local), or remote with empty user, host or remotepath
    print('dest should use the following format: user@192.168.0.2:/path/to/backup/')
    exit(2)
print('Starting backup...\nSource path: %s\nDestination host: %s\nDestination path: %s' % (src, host, remotepath))
if sftppwd is None:
    sftppwd = getpass.getpass('Please enter the SFTP password for user %s: ' % user)
if encryptionpwd is None:
    encryptionpwd = getpass.getpass('Please enter the encryption password: ')
key, salt = KDF(encryptionpwd)        
try:
    with pysftp.Connection(host, username=user, password=sftppwd) as sftp:
        print(sftp.getcwd())
        print(sftp.listdir())
        if sftp.isdir(remotepath):
            sftp.chdir(remotepath)
        else:    
            print('Destination directory does not exist.')
            exit(2)
except paramiko.ssh_exception.AuthenticationException:
    print('Authentication failed.')
except paramiko.ssh_exception.SSHException as e:
    print(e, '\nPlease ssh your remote host at least once before, or add your remote to your known_hosts file.\n\n')  # todo: avoid ugly error messages after


True admin 192.168.0.25 ./usbshare1-2/benoit/backup-nas/Divers/
Starting backup...
Source path: /home/ben/Desktop
Destination host: 192.168.0.25
Destination path: ./usbshare1-2/benoit/backup-nas/Divers/
None
['Backup', 'Divers', 'MailPlus', 'MaisonMarieClaire', 'NetBackup', 'OHAPE', 'Partage', 'chat', 'docker', 'home', 'homes', 'music', 'photo', 'surveillance', 'usbshare1-1', 'usbshare1-2', 'video', 'web']
